# RNNs & LSTMs (Brief) — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



6 tasks, 2 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

## Task 1: Dataset & Setup

# RNNs & LSTMs (Brief) — Student Lab

You will implement a tiny character-level next-token model in NumPy:
- vanilla RNN forward + BPTT
- gradient clipping
- LSTM cell

Everything is offline and deterministic.

## Section 0 — Data (character-level)
We build a small repeated-pattern corpus so the model has learnable structure.

## Utilities
Stable softmax cross-entropy for a batch.

In [ ]:
# No source solution — left as an exercise.

## Task 2: Implement one-hot encoding and the vanilla RNN forward pass.
*Section: Vanilla RNN forward*

## Section 1 - Vanilla RNN forward

We use a simple RNN: `h_t = tanh(x_t Wxh + h_{t-1} Whh + bh)` and logits `h_t Why + by`.

### Task 1.1
Implement `one_hot` and `rnn_forward` returning:
- `hs`: `(T, B, H)`
- `logits`: `(T, B, V)`
- `cache['xs']`: one-hot inputs used later by BPTT.

**Hints**
- Build the one-hot tensor with advanced indexing, then iterate through time to update `h_t`.

In [ ]:
$$12

In [ ]:
# Checks
check('rnn_shapes', hs.shape == (seq_len, batch_size, H) and logits.shape == (seq_len, batch_size, V) and cache['xs'].shape == (seq_len, batch_size, V))

**Why this works.** ## Vanilla RNN forward
Rationale: the hidden state compresses prefix information one step at a time, and the cached one-hot inputs are needed later for BPTT.

## Task 3: Implement BPTT for the vanilla RNN loss and gradients.
*Section: BPTT (vanilla RNN)*

## Section 2 - BPTT (vanilla RNN)

### Task 2.1
Implement `clip_grads` and `rnn_loss_and_grads`.
Compute average cross-entropy across time, then backpropagate through the unrolled recurrence.

Return `(loss, grads)` where `grads` has keys `Wxh`, `Whh`, `Why`, `bh`, `by`.

Hint: backprop `tanh` with `(1 - h^2)` and accumulate `dh_next` backward through time.

**Hints**
- Unroll backward from `t = T-1` to `0`; the recurrent gradient contribution arrives through `dh_next`.

In [ ]:
# No source solution — left as an exercise.

## Task 4: Train the vanilla RNN briefly and inspect gradient clipping.
*Section: Training loop + gradient diagnostics*

## Section 3 - Training loop + gradient diagnostics

### Task 3.1
Train the vanilla RNN for a few steps and verify loss decreases.
Also print gradient norm before clipping and after clipping so you can see what clipping changes during optimization.

In [ ]:
$10

In [ ]:
# Checks
check('loss_down', losses[-1] < losses[0])
check('clip_nonincreasing', max(grad_after) <= max(grad_before) + 1e-9)

**Why this works.** ## Training loop + gradient clipping
Rationale: clipping does not solve optimization by itself; it prevents unstable updates when recurrent gradients spike during BPTT.

## Task 5: Implement an LSTM forward pass and an output-head-only loss function.
*Section: LSTM cell (forward + output-head-only demo)*

## Section 4 - LSTM cell (forward + output-head-only demo)

### Task 4.1
Implement an LSTM forward pass and a brief loss function on the same data.
In this lab, we do **not** implement full BPTT through the LSTM gates.
We only compute gradients for the output projection `Why, by` so the section stays focused on gate equations and stable cell-state flow. That means this section explains why LSTMs help memory, but it is not yet teaching full recurrent optimization through the gates.

Hint: gates use sigmoid; candidate uses tanh; cell state accumulates through `c_t = f_t * c_{t-1} + i_t * g_t`.

**Hints**
- Split the activation block into forget, input, output, and candidate chunks of width `H`.

In [ ]:
$$17

In [ ]:
# Checks
check('lstm_brief_shapes', hs.shape == (seq_len, batch_size, H) and logits.shape == (seq_len, batch_size, V) and np.isfinite(loss0))

**Why this works.** ## LSTM forward + output-head-only loss
Rationale: the LSTM gate equations are the main concept here. Full gate-level BPTT is intentionally deferred; only the output head is trained in this brief lab.

## Task 6: Train only the LSTM output layer briefly and compare loss trend.
*Section: LSTM cell (forward + output-head-only demo)*

### Task 4.2
Train only the LSTM output layer briefly and compare loss trend.
Because the recurrent LSTM parameters are frozen in this lab, expect improvement to be slower than the fully trained vanilla RNN section. Read the result as an output-head demo, not as complete LSTM training.

In [ ]:
params = {k: v.copy() for k, v in params_lstm.items()}
losses = []
for it in range(80):
    Xb, Yb = batches[it % len(batches)]
    loss, grads = lstm_loss_and_grads(Xb, Yb, params, h0_lstm, c0_lstm)
    sgd_step(params, grads, lr=0.8)
    losses.append(loss)
    if (it+1) % 20 == 0:
        print('it', it+1, 'loss', round(loss, 4))

check('lstm_loss_down', losses[-1] < losses[0])

In [ ]:
# Checks
check('lstm_loss_down', losses[-1] < losses[0])

**Why this works.** ## Output-head-only LSTM training demo
Rationale: this section is a controlled demo showing that better recurrent state dynamics and actual recurrent optimization are different topics.